# Segmentation des graines de blé — K-means et CAH

UCI Seeds contient 210 graines de blé décrites par sept mesures géométriques. Le fichier fourni contient ces sept mesures sans les étiquettes de variétés.

**CSV à placer à côté du notebook :** `wheat_seeds_dataset.csv`  
**Source :** [UCI Machine Learning Repository — Seeds](https://archive.ics.uci.edu/dataset/236/seeds)

### Objectifs

Examiner et nettoyer les données, appliquer K-means et la classification ascendante hiérarchique (CAH), comparer les groupes, puis expliquer les limites de l’analyse.

## 1. Charger les bibliothèques et les données

Les colonnes décrivent l’aire, le périmètre, la compacité, la longueur et la largeur du grain, le coefficient d’asymétrie et la longueur du sillon.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("wheat_seeds_dataset.csv")
df.head()

## 2. Examiner les données

`shape` indique le nombre de lignes et de colonnes, `info` affiche les types et valeurs présentes, et `describe` résume les colonnes numériques.

In [ ]:
print("Dimensions initiales :", df.shape)
df.info()
df.describe().T

## 3. Nettoyer les données

Les marqueurs manquants sont convertis en valeurs manquantes, les lignes et colonnes entièrement vides sont retirées, les autres valeurs manquantes sont remplacées par la médiane. Les colonnes constantes sont retirées car elles ne séparent pas les observations.

In [ ]:
df = df.replace(["?", "NA", "N/A", ""], np.nan)
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(how="all").reset_index(drop=True)
df = df.dropna(axis=1, how="all")
X_clean = df.select_dtypes(include="number").copy()
X_clean = X_clean.loc[:, X_clean.nunique(dropna=True) > 1]
X_clean = X_clean.fillna(X_clean.median())
if X_clean.shape[1] < 2:
    raise ValueError("Il faut au moins deux variables numériques exploitables.")
print("Dimensions après nettoyage :", X_clean.shape)
print("Valeurs manquantes restantes :", int(X_clean.isna().sum().sum()))
X_clean.head()

## 4. Standardiser les variables

Les deux méthodes utilisent des distances. La standardisation évite qu’une colonne ayant de grandes valeurs domine les autres.

In [ ]:
from sklearn.preprocessing import normalize
x = normalize(X_clean.to_numpy(dtype=float))

## 5. Projeter les données en deux dimensions avec une ACP

L’ACP sert à visualiser les observations. Les algorithmes de clustering utilisent toujours les variables standardisées originales.

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
projection = pca.fit_transform(x)
New_df = pd.DataFrame(projection, columns=["PCA1", "PCA2"])
print(f"Variance expliquée par les deux axes : {pca.explained_variance_ratio_.sum()*100:.2f}%")

## 6. Examiner le nombre de groupes avec la méthode du coude

Le coude compare l’inertie pour plusieurs valeurs de *k*. Il donne une indication, à compléter par l’analyse des silhouettes et des profils de groupes.

In [ ]:
from sklearn.cluster import KMeans
from yellowbrick.cluster import KElbowVisualizer
visualizer = KElbowVisualizer(KMeans(init="random", max_iter=500, n_init=10, random_state=42), k=(2, 10))
visualizer.fit(x)
visualizer.show()

### Choix du nombre de groupes

La source UCI présente trois variétés, mais le CSV local ne contient pas les étiquettes. On examine le coude puis on utilise trois groupes comme hypothèse de départ. On ne peut pas attribuer un nom de variété aux groupes sans les étiquettes.

## 7. K-means avec k = 3

Trois groupes servent d’hypothèse de départ car le jeu UCI décrit trois variétés; vérifie ce choix avec le coude et les silhouettes.

In [ ]:
from sklearn.metrics import silhouette_score
k = 3
kmeans_model = KMeans(n_clusters=k, init="random", max_iter=500, n_init=10, random_state=42)
labels_kmeans = kmeans_model.fit_predict(x)
sil_KM = silhouette_score(x, labels_kmeans, sample_size=min(1000, len(x)), random_state=42)
print(pd.Series(labels_kmeans, name="Groupe").value_counts().sort_index())
print(f"Score de silhouette K-means : {sil_KM:.3f}")

In [ ]:
px.scatter(New_df, x="PCA1", y="PCA2", color=labels_kmeans.astype(str), title="K-means — projection ACP", labels={"color":"Groupe"}).show()

## 8. Classification ascendante hiérarchique (CAH), lien average

Le dendrogramme montre les fusions successives. Il est tronqué pour rester lisible. La CAH est ensuite coupée en 3 groupes pour comparer avec K-means.

In [ ]:
from scipy.cluster.hierarchy import linkage as hierarchy_linkage, dendrogram
Z = hierarchy_linkage(x, method="average")
plt.figure(figsize=(13, 6))
dendrogram(Z, truncate_mode="lastp", p=30, show_leaf_counts=True)
plt.title("Dendrogramme CAH — average")
plt.xlabel("Groupes fusionnés")
plt.ylabel("Distance")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import AgglomerativeClustering
cah_model = AgglomerativeClustering(n_clusters=3, linkage="average")
labels_cah = cah_model.fit_predict(x)
sil_CAH = silhouette_score(x, labels_cah, sample_size=min(1000, len(x)), random_state=42)
print(pd.Series(labels_cah, name="Groupe").value_counts().sort_index())
print(f"Score de silhouette CAH : {sil_CAH:.3f}")
px.scatter(New_df, x="PCA1", y="PCA2", color=labels_cah.astype(str), title="CAH — projection ACP", labels={"color":"Groupe"}).show()

## 9. Comparer et interpréter les résultats

Le score de silhouette varie de -1 à 1. Un score plus élevé suggère des groupes mieux séparés; un score proche de 0 indique du chevauchement. Il faut aussi examiner les graphiques et les profils moyens.

In [ ]:
comparaison = pd.DataFrame({"Méthode": ["K-means", "CAH"], "Silhouette": [sil_KM, sil_CAH]})
comparaison

In [ ]:
px.bar(comparaison, x="Méthode", y="Silhouette", color="Méthode", text_auto=".3f", title="Scores de silhouette").show()

In [ ]:
profil_kmeans = X_clean.assign(Groupe=labels_kmeans).groupby("Groupe").mean().round(2)
profil_cah = X_clean.assign(Groupe=labels_cah).groupby("Groupe").mean().round(2)
print("Profils moyens — K-means")
display(profil_kmeans)
print("Profils moyens — CAH")
display(profil_cah)

## Conclusion — aide pour ta vidéo

Présente les données, le nettoyage, l’intérêt de la standardisation, le choix de *k*, les silhouettes et les profils de groupes. Explique que les numéros de groupe sont arbitraires et que l’interprétation doit venir des variables qui distinguent les profils. Les étiquettes des variétés ne sont pas fournies; les groupes ne peuvent donc pas être validés par variété.